# 05 — Ablation matrix (RQ2, guide §7.3) — SMOKE dry run

The full 8-condition × 5-fold harness at ultra-tiny settings (1 seed, 1 epoch,
6 steps/epoch, short sequences). This proves the loop, the per-condition
artifacts, and the significance tests end-to-end. **Every number is SMOKE** —
with ~1–2 test positives per fold the metrics are noise by construction (§11.6).

Disabled channels are REMOVED from the gate softmax (renormalized), not
zero-filled — `model.py` builds the gate only over active channels.

In [1]:
import time

import pandas as pd

from leische import data as D
from leische import evaluate as E
from leische import train as T
from leische.config import LeischeConfig, find_repo_root
from leische.nbsupport import bootstrap

cfg0, df, corpus, card, gate = bootstrap(config="configs/ablation_smoke.yaml", need_corpus=True)
SMOKE = cfg0.tag()
folds = D.load_folds(cfg0.folds_file(), card)
root = find_repo_root()

# the §7.3 matrix
CONDITIONS = [
    ("1_baseline", False, False, False),
    ("2_conv", True, False, False),
    ("3_temp", False, True, False),
    ("4_ret", False, False, True),
    ("5_conv_temp", True, True, False),
    ("6_conv_ret", True, False, True),
    ("7_temp_ret", False, True, True),
    ("8_full", True, True, True),
]

C:\Users\Gio\Documents\GitHub\leische\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 SMOKE MODE — every number in this notebook is a harness or
 architecture check on the pilot export, NOT a reportable result.
dataset identity: {'dataset_version': 'v1', 'prompt_version': 'sarc-v1', 'uyam_commit': '69d0affaca157edaf64e617d21e8ae752c9343fa'}
rows: 100  sarcastic: 8  corpus: loaded

§10 data-readiness gate:
  [FAIL] dataset-v2 under sarc-v2 — dataset_version=v1, prompt_version=sarc-v1
  [FAIL] ≥400 sarcastic positives — 8 positives in export
  [FAIL] gold subset labeled + κ reported — n_gold_items=0, sarcastic κ=None
  [FAIL] every language×sarcastic cell ≥10 — thin cells: {'taglish|sarc=True': 5, 'english|sarc=True': 2, 'tagalog|sarc=True': 1}
  [PASS] fold file frozen — C:\Users\Gio\Documents\GitHub\leische\results\folds-v1.json
  => GATE FAILED: notebooks run in SMOKE mode only; no reported metrics.


## Run all 8 conditions × 5 folds (× 1 seed, tiny settings)

In [2]:
results = {}
for name, use_conv, use_temp, use_ret in CONDITIONS:
    t0 = time.time()
    cfg = LeischeConfig.from_yaml(root / "configs/ablation_smoke.yaml",
                                  use_conv=use_conv, use_temp=use_temp, use_ret=use_ret,
                                  run_name=f"ablation-smoke-{name}")
    results[name] = T.run_cv(cfg, df, corpus, card, folds, verbose=False)
    fm = results[name]["fold_metrics"]
    print(f"{SMOKE}{name:<12} conv={int(use_conv)} temp={int(use_temp)} ret={int(use_ret)} "
          f"| mean F1 {fm['f1'].mean():.3f} | {time.time() - t0:5.1f}s "
          f"({len(fm)} fold-runs)")

SMOKE 1_baseline   conv=0 temp=0 ret=0 | mean F1 0.000 |  13.3s (5 fold-runs)


SMOKE 2_conv       conv=1 temp=0 ret=0 | mean F1 0.000 |  14.2s (5 fold-runs)


SMOKE 3_temp       conv=0 temp=1 ret=0 | mean F1 0.181 |  14.1s (5 fold-runs)


SMOKE 4_ret        conv=0 temp=0 ret=1 | mean F1 0.050 |  20.8s (5 fold-runs)


SMOKE 5_conv_temp  conv=1 temp=1 ret=0 | mean F1 0.000 |  15.4s (5 fold-runs)


SMOKE 6_conv_ret   conv=1 temp=0 ret=1 | mean F1 0.000 |  28.7s (5 fold-runs)


SMOKE 7_temp_ret   conv=0 temp=1 ret=1 | mean F1 0.069 |  23.1s (5 fold-runs)


SMOKE 8_full       conv=1 temp=1 ret=1 | mean F1 0.050 |  33.6s (5 fold-runs)


## Ablation table (mean ± std across folds)

In [3]:
table = E.ablation_table({n: r["fold_metrics"] for n, r in results.items()})
print(f"{SMOKE}RQ2 ablation matrix — HARNESS CHECK ONLY, pilot metrics are noise:")
print(table.to_string(index=False))
csv_path = cfg0.results_path() / "SMOKE-ablation-matrix.csv"
table.to_csv(csv_path, index=False)
print(f"\nsaved → {csv_path}")

SMOKE RQ2 ablation matrix — HARNESS CHECK ONLY, pilot metrics are noise:
  condition  runs              f1       precision          recall        accuracy  mean_test_pos
 1_baseline     5 0.0000 ± 0.0000 0.0000 ± 0.0000 0.0000 ± 0.0000 0.9204 ± 0.0554            1.6
     2_conv     5 0.0000 ± 0.0000 0.0000 ± 0.0000 0.0000 ± 0.0000 0.9204 ± 0.0554            1.6
     3_temp     5 0.1808 ± 0.1093 0.1024 ± 0.0628 0.8000 ± 0.4472 0.3223 ± 0.1244            1.6
      4_ret     5 0.0500 ± 0.1118 0.0286 ± 0.0639 0.2000 ± 0.4472 0.7775 ± 0.3573            1.6
5_conv_temp     5 0.0000 ± 0.0000 0.0000 ± 0.0000 0.0000 ± 0.0000 0.9204 ± 0.0554            1.6
 6_conv_ret     5 0.0000 ± 0.0000 0.0000 ± 0.0000 0.0000 ± 0.0000 0.9204 ± 0.0554            1.6
 7_temp_ret     5 0.0690 ± 0.1092 0.0386 ± 0.0622 0.4000 ± 0.5477 0.3975 ± 0.4592            1.6
     8_full     5 0.0500 ± 0.1118 0.0286 ± 0.0639 0.2000 ± 0.4472 0.7775 ± 0.3573            1.6

saved → C:\Users\Gio\Documents\GitHub\leische\results

## Significance tests: condition 8 (full) vs condition 1 (baseline)

Paired bootstrap + approximate randomization on shared test predictions,
plus McNemar for the headline pair (guide §7.3). Wired now, meaningful
only post-gate.

In [4]:
pred_base = results["1_baseline"]["predictions"]
pred_full = results["8_full"]["predictions"]

boot = E.paired_bootstrap(pred_base, pred_full, n_boot=1000, seed=13)
print(f"{SMOKE}paired bootstrap ΔF1 (full − baseline): {boot['observed_delta_f1']:+.4f} "
      f"CI95 [{boot['ci95'][0]:+.4f}, {boot['ci95'][1]:+.4f}] p={boot['p_two_sided']:.3f} "
      f"(n={boot['n_rows']})")

ar = E.approximate_randomization(pred_base, pred_full, n_iter=1000, seed=13)
print(f"{SMOKE}approximate randomization |ΔF1|={ar['observed_abs_delta_f1']:.4f} "
      f"p={ar['p_value']:.3f}")

mc = E.mcnemar_test(pred_base, pred_full)
print(f"{SMOKE}McNemar (correct/incorrect discordants {mc['table'][0][1]} vs "
      f"{mc['table'][1][0]}): p={mc['p_value']:.3f}")
print(f"\n{SMOKE}with ~8 positives these p-values are definitionally meaningless — "
      "the deliverable is that the machinery runs.")

SMOKE paired bootstrap ΔF1 (full − baseline): +0.2069 CI95 [+0.0000, +0.4211] p=0.098 (n=100)


SMOKE approximate randomization |ΔF1|=0.2069 p=0.262
SMOKE McNemar (correct/incorrect discordants 18 vs 3): p=0.001

SMOKE with ~8 positives these p-values are definitionally meaningless — the deliverable is that the machinery runs.


## §7.2 hygiene + diagnostics for the full condition

In [5]:
print(f"{SMOKE}natural-only vs all rows (condition 8):")
print(E.natural_and_all(pred_full).to_string(index=False))
print("\npilot has 0 keyword_oversampled rows → the two slices coincide; the "
      "filter path is what's being exercised.")
print(f"\n{SMOKE}condition-8 F1 by resolved_by (is the model only right on easy "
      "unanimous items? — §7.4):")
print(E.slice_metrics(pred_full, "resolved_by").to_string(index=False))
print(f"\n{SMOKE}condition-8 mean gates by language:")
print(E.gate_summary(pred_full, by="language").to_string(index=False))

SMOKE natural-only vs all rows (condition 8):
       slice       f1  precision  recall  accuracy   n  n_pos
natural_only 0.206897   0.142857   0.375      0.77 100      8
    all_rows 0.206897   0.142857   0.375      0.77 100      8

pilot has 0 keyword_oversampled rows → the two slices coincide; the filter path is what's being exercised.

SMOKE condition-8 F1 by resolved_by (is the model only right on easy unanimous items? — §7.4):
resolved_by       f1  precision   recall  accuracy  n  n_pos
adjudicator 0.000000   0.000000 0.000000  0.941176 17      0
   majority 0.181818   0.133333 0.285714  0.694915 59      7
  unanimous 0.333333   0.200000 1.000000  0.833333 24      1

SMOKE condition-8 mean gates by language:
language  gate_conv  gate_temp  gate_ret
 english   0.521488   0.161555  0.316958
 tagalog   0.498658   0.176257  0.325085
 taglish   0.489851   0.181422  0.328726
